# SRA Metadata Ingestion Runner

This notebook demonstrates how to use the `SRAAccessor` to ingest and validate SRA metadata from a CSV file. It's a practical example of our iDesign architecture in action, where data is fetched and validated against a strict contract at the system's entry point.

In [1]:
# Imports
import sys
import os
import pandas as pd
from pathlib import Path
from rich import print

# Add the project's root directory to the system path.
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from Accessors.SraAccessor.sra_metadata_accessor import SRAMetadataAccessor
from Accessors.SraAccessor.sra_taxonomy_accessor import SRATaxonomyAccessor
from processors.sra_processor import SRAProcessor


In [2]:
# Define the view folder containing metadata.csv and taxonomy.csv.
view_path = Path("Inputs/PhagesInBlood")


In [3]:
# Load and validate the two source files using the new accessor classes.
try:
    metadata_df = SRAMetadataAccessor.get(view_path / "metadata.csv")
    taxonomy_df = SRATaxonomyAccessor.get(view_path / "taxonomy.csv")

    print(f"✅ Loaded metadata: {len(metadata_df)} samples")
    print(f"✅ Loaded taxonomy: {len(taxonomy_df)} taxon observations across {taxonomy_df['acc'].nunique()} samples")

    with pd.option_context('display.max_rows', 5, 'display.max_columns', None):
        display(metadata_df.head())

except Exception as e:
    print(f"❌ An error occurred: {e}")


✅ Loaded metadata: 139 samples

✅ Loaded taxonomy: 46236 taxon observations across 139 samples

,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,platform,sample_acc,biosample,organism,sra_study,releasedate,mbytes,loaddate,avgspotlen,mbases,insertsize,library_name,biosamplemodel_sam,collection_date_sam,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,ena_first_public_run,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject
0,SRR16574708,OTHER,GEO,public,SRX12776662,GSM5655983,HiSeq X Ten,PAIRED,other,GENOMIC,ILLUMINA,SRS10721818,SAMN22567976,Homo sapiens,SRP343169,2023-09-28,597,NaN,300,1990,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[fastq, run.zq, sra]","[s3, ncbi, gs]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655983}, {k=bases...","{""geo_accession_exp"": [""GSM5655983""], ""bases"":...",1,PRJNA774626
1,SRR16574682,OTHER,GEO,public,SRX12776636,GSM5655957,HiSeq X Ten,PAIRED,other,GENOMIC,ILLUMINA,SRS10721792,SAMN22568002,Homo sapiens,SRP343169,2023-09-28,1635,NaN,300,5507,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=geo_accession_exp, v=GSM5655957}, {k=bases...","{""geo_accession_exp"": [""GSM5655957""], ""bases"":...",1,PRJNA774626
2,SRR16574640,OTHER,GEO,public,SRX12776594,GSM5655915,HiSeq X Ten,PAIRED,other,GENOMIC,ILLUMINA,SRS10721750,SAMN22568020,Homo sapiens,SRP343169,2023-09-28,6040,NaN,300,19915,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[run.zq, sra, fastq]","[s3, gs, ncbi]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655915}, {k=bases...","{""geo_accession_exp"": [""GSM5655915""], ""bases"":...",1,PRJNA774626
3,SRR16574660,OTHER,GEO,public,SRX12776614,GSM5655935,HiSeq X Ten,PAIRED,other,GENOMIC,ILLUMINA,SRS10721770,SAMN22568040,Homo sapiens,SRP343169,2023-09-28,1062,NaN,300,3526,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[fastq, sra, run.zq]","[gs, s3, ncbi]","[gs.us-east1, s3.us-east-1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655935}, {k=bases...","{""geo_accession_exp"": [""GSM5655935""], ""bases"":...",1,PRJNA774626
4,SRR16574696,OTHER,GEO,public,SRX12776650,GSM5655971,HiSeq X Ten,PAIRED,other,GENOMIC,ILLUMINA,SRS10721806,SAMN22568046,Homo sapiens,SRP343169,2023-09-28,1706,NaN,300,5668,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[sra, run.zq, fastq]","[ncbi, s3, gs]","[s3.us-east-1, gs.us-east1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655971}, {k=bases...","{""geo_accession_exp"": [""GSM5655971""], ""bases"":...",1,PRJNA774626


In [4]:
# --- Step 4: Discover Projects and Generate/Refresh Templates ---
# Scans metadata and creates (or refreshes) configuration templates.
# Templates are always refreshed so discovered keys & value counts stay current.
# Existing .json configs are NEVER touched — only .json.template files are updated.

from pathlib import Path

unique_project_ids = metadata_df['bioproject'].unique()
print(f"🔬 Found {len(unique_project_ids)} unique project(s): {list(unique_project_ids)}")

sra_processor = SRAProcessor()
config_dir = Path("mapping_configs")

projects_needing_config = []
projects_ready = []

for project_id in unique_project_ids:
    config_path = config_dir / f"{project_id}.json"
    project_df = metadata_df[metadata_df['bioproject'] == project_id].copy()

    # Always refresh the template (safe — never touches .json configs)
    sra_processor.refresh_template(project_df, project_id)

    if config_path.exists():
        projects_ready.append(project_id)
        print(f"✅ [bold]{project_id}[/bold]: Configuration ready (template refreshed)")
    else:
        projects_needing_config.append(project_id)
        print(f"📝 [bold]{project_id}[/bold]: Template generated, needs configuration")

print(f"\n{'='*50}")
if projects_needing_config:
    print(f"⏳ {len(projects_needing_config)} project(s) need configuration: {projects_needing_config}")
    print("   → Run the next cell to configure mappings interactively")
else:
    print(f"✅ All {len(projects_ready)} project(s) are configured!")
    print("   → Run the next cell to review/update mappings, or skip to Step 6")


🔬 Found 4 unique project(s): ['PRJNA774626', 'PRJEB89352', 'PRJEB64933', 'PRJEB50557']

✅ PRJNA774626: Configuration ready (template refreshed)

✅ PRJEB89352: Configuration ready (template refreshed)

✅ PRJEB64933: Configuration ready (template refreshed)

✅ PRJEB50557: Configuration ready (template refreshed)

==================================================

✅ All 4 project(s) are configured!

→ Run the next cell to review/update mappings, or skip to Step 6

In [5]:
# --- Step 5: Interactive Mapping Configuration (Target-First Approach) ---
# This cell provides a dynamic GUI to create JSON mapping files with a "target field first" workflow.
#
# How to use:
# 1. Run this cell. The UI shows all existing target fields configured across projects.
# 2. Expand "🔍 Discovered Source Keys" to see available metadata fields from source data.
# 3. Click "+ Add Target Field" to add a new Metalog field you want to map.
# 4. For each target field panel:
#    - Configure how each PROJECT maps to this target (one row per project)
#    - Choose mode: 🔑 Source (map from a key), 📌 Static (same value for all), ⏭️ Skip
#    - Click "📝 Values" to define value transformations if needed
#    - Click "💾 Save" to save this target field's mappings to all project configs
# 5. Run Step 6 to apply your mappings and prepare data for analysis.

from managers.MappingConfigManager.mapping_config_manager import MappingConfigManager

# Create and run the mapping configuration manager
mapping_manager = MappingConfigManager()
mapping_manager.run()

✅ Loaded 27 fields from Metalog schema
📋 Loaded 4 project(s)
📝 Found 5 existing target field(s): ['Group', 'environment_material', 'sex', 'subject_id', 'timepoint']



In [6]:
# --- Step 6: Apply Mappings & Prepare Analysis-Ready Dataset ---
# Applies configured mappings to metadata only (no taxonomy columns involved).
# Produces enriched_metadata_df: 1 row per acc, with mapped fields (sex, Group, etc.) added.

from pathlib import Path

config_dir = Path("mapping_configs")
sra_processor = SRAProcessor()

processed_parts = []
missing_configs = []

for project_id in metadata_df['bioproject'].unique():
    config_path = config_dir / f"{project_id}.json"

    if not config_path.exists():
        missing_configs.append(project_id)
        continue

    print(f"📊 Processing [bold]{project_id}[/bold]...")
    project_df = metadata_df[metadata_df['bioproject'] == project_id].copy()
    processed_df = sra_processor.process(project_df, project_id)
    processed_parts.append(processed_df)

if missing_configs:
    print(f"\n⚠️ Missing configurations for: {missing_configs}")
    print("   → Run Steps 4-5 to configure these projects first")
elif not processed_parts:
    print("❌ No data to process. Check Steps 1-5.")
else:
    enriched_metadata_df = pd.concat(processed_parts, ignore_index=True)

    from common.contracts.sra_schema import SRAMetadataSchema
    base_cols = set(SRAMetadataSchema.__fields__.keys())
    mapped_cols = [c for c in enriched_metadata_df.columns if c not in base_cols]

    print(f"\n{'='*60}")
    print(f"✅ [bold green]Enriched Metadata Ready![/bold green]")
    print(f"{'='*60}")
    print(f"📊 Total samples: {len(enriched_metadata_df)}")
    print(f"🔗 Projects: {enriched_metadata_df['bioproject'].nunique()}")
    print(f"📝 Mapped fields: {mapped_cols if mapped_cols else 'None (only filters applied)'}")

    print(f"\n{'─'*60}")
    print("📈 [bold]Grouping Variables:[/bold]")
    for col in mapped_cols:
        if enriched_metadata_df[col].dtype == 'object':
            value_counts = enriched_metadata_df[col].value_counts(dropna=False)
            print(f"\n  [bold]{col}[/bold] ({len(value_counts)} unique values):")
            for val, count in value_counts.head(5).items():
                print(f"    • {val}: {count} samples")
            if len(value_counts) > 5:
                print(f"    ... and {len(value_counts) - 5} more")

    display(enriched_metadata_df.head())


📊 Processing PRJNA774626...

✅ Found configuration for project PRJNA774626. Applying mappings...

📊 Processing PRJEB89352...

✅ Found configuration for project PRJEB89352. Applying mappings...

🚫 Project excluded by filter configuration

...

🔗 Projects: 1

📝 Mapped fields: ['environment_material', 'sex', 'Group']

────────────────────────────────────────────────────────────

📈 Grouping Variables:

,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject,environment_material,sex,Group
0,SRR16574708,OTHER,GEO,public,SRX12776662,GSM5655983,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, run.zq, sra]","[s3, ncbi, gs]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655983}, {k=bases...","{""geo_accession_exp"": [""GSM5655983""], ""bases"":...",1,PRJNA774626,plasma,male,Colon Cancer
1,SRR16574682,OTHER,GEO,public,SRX12776636,GSM5655957,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=geo_accession_exp, v=GSM5655957}, {k=bases...","{""geo_accession_exp"": [""GSM5655957""], ""bases"":...",1,PRJNA774626,plasma,female,Colon Cancer
2,SRR16574640,OTHER,GEO,public,SRX12776594,GSM5655915,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[run.zq, sra, fastq]","[s3, gs, ncbi]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655915}, {k=bases...","{""geo_accession_exp"": [""GSM5655915""], ""bases"":...",1,PRJNA774626,plasma,female,Colon Cancer
3,SRR16574660,OTHER,GEO,public,SRX12776614,GSM5655935,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, sra, run.zq]","[gs, s3, ncbi]","[gs.us-east1, s3.us-east-1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655935}, {k=bases...","{""geo_accession_exp"": [""GSM5655935""], ""bases"":...",1,PRJNA774626,plasma,female,healthy
4,SRR16574696,OTHER,GEO,public,SRX12776650,GSM5655971,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[sra, run.zq, fastq]","[ncbi, s3, gs]","[s3.us-east-1, gs.us-east1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655971}, {k=bases...","{""geo_accession_exp"": [""GSM5655971""], ""bases"":...",1,PRJNA774626,plasma,male,Colon Cancer


In [7]:
# --- Step 7: Build Canonical Wide Analysis Dataset ---
#
# Creates one canonical table used by all downstream analyses.
# Rows = samples (acc), columns = metadata fields + taxon features (rank|name|tax_id).
import base64
import pandas as pd
from IPython.display import HTML, display

if 'enriched_metadata_df' not in locals() or 'taxonomy_df' not in locals():
    raise RuntimeError("Run Steps 3 and 6 first to create taxonomy_df and enriched_metadata_df")

TAXON_KEY_SEP = '||'

# Keep a stable list of metadata columns so downstream cells can reconstruct metadata from the wide table.
metadata_cols = list(enriched_metadata_df.columns)

# Keep lineage columns (ileft/iright) so managers that depend on taxonomy hierarchy still work.
lineage_cols = [c for c in ['ileft', 'iright'] if c in taxonomy_df.columns]

# Build a unique taxon key that preserves rank + name + tax_id.
taxonomy_for_wide = taxonomy_df[['acc', 'tax_id', 'rank', 'name', 'total_count'] + lineage_cols].copy()
taxonomy_for_wide['taxon_key'] = (
    taxonomy_for_wide['rank'].astype(str) + TAXON_KEY_SEP +
    taxonomy_for_wide['name'].astype(str) + TAXON_KEY_SEP +
    taxonomy_for_wide['tax_id'].astype(str)
)

taxon_lookup_cols = ['taxon_key', 'rank', 'name', 'tax_id'] + lineage_cols
taxon_lookup = taxonomy_for_wide[taxon_lookup_cols].drop_duplicates(subset=['taxon_key'])

taxon_wide = taxonomy_for_wide.pivot_table(
    index='acc',
    columns='taxon_key',
    values='total_count',
    aggfunc='sum',
    fill_value=0
)

analysis_wide_df = enriched_metadata_df.merge(
    taxon_wide.reset_index(),
    on='acc',
    how='left'
)

# Fill missing taxon columns with 0 for samples that have no counts for a given taxon.
taxon_cols = [c for c in analysis_wide_df.columns if c not in metadata_cols]
analysis_wide_df[taxon_cols] = analysis_wide_df[taxon_cols].fillna(0)

print(f"✅ Canonical analysis_wide_df ready: {analysis_wide_df.shape[0]} samples × {analysis_wide_df.shape[1]} columns")
print(f"🧬 Taxon feature columns: {len(taxon_cols)}")
print(f"🌳 Lineage columns preserved: {lineage_cols if lineage_cols else 'None'}")
display(analysis_wide_df[metadata_cols].head())

# Optional CSV export of taxon-only wide matrix for external tools.
feature_matrix = analysis_wide_df[['acc'] + taxon_cols].set_index('acc')
csv_data = feature_matrix.to_csv()
b64_data = base64.b64encode(csv_data.encode()).decode()
payload = f"data:text/csv;base64,{b64_data}"

html_button = f'''
<a download="Ask_SLM_Taxon_Abundance_Matrix.csv" href="{payload}" target="_blank">
    <button style="background-color: #4CAF50; color: white; padding: 12px 24px; border: none; border-radius: 6px; cursor: pointer; font-weight: bold; font-size: 14px;">
        Download Abundance Matrix (CSV)
    </button>
</a>
'''
display(HTML(html_button))

✅ Canonical analysis_wide_df ready: 86 samples × 4309 columns

🧬 Taxon feature columns: 4269

🌳 Lineage columns preserved: ['ileft', 'iright']

,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject,environment_material,sex,Group
0,SRR16574708,OTHER,GEO,public,SRX12776662,GSM5655983,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, run.zq, sra]","[s3, ncbi, gs]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655983}, {k=bases...","{""geo_accession_exp"": [""GSM5655983""], ""bases"":...",1,PRJNA774626,plasma,male,Colon Cancer
1,SRR16574682,OTHER,GEO,public,SRX12776636,GSM5655957,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=geo_accession_exp, v=GSM5655957}, {k=bases...","{""geo_accession_exp"": [""GSM5655957""], ""bases"":...",1,PRJNA774626,plasma,female,Colon Cancer
2,SRR16574640,OTHER,GEO,public,SRX12776594,GSM5655915,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[run.zq, sra, fastq]","[s3, gs, ncbi]","[ncbi.public, s3.us-east-1, gs.us-east1]","[{k=geo_accession_exp, v=GSM5655915}, {k=bases...","{""geo_accession_exp"": [""GSM5655915""], ""bases"":...",1,PRJNA774626,plasma,female,Colon Cancer
3,SRR16574660,OTHER,GEO,public,SRX12776614,GSM5655935,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[fastq, sra, run.zq]","[gs, s3, ncbi]","[gs.us-east1, s3.us-east-1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655935}, {k=bases...","{""geo_accession_exp"": [""GSM5655935""], ""bases"":...",1,PRJNA774626,plasma,female,healthy
4,SRR16574696,OTHER,GEO,public,SRX12776650,GSM5655971,HiSeq X Ten,PAIRED,other,GENOMIC,...,"[sra, run.zq, fastq]","[ncbi, s3, gs]","[s3.us-east-1, gs.us-east1, ncbi.public]","[{k=geo_accession_exp, v=GSM5655971}, {k=bases...","{""geo_accession_exp"": [""GSM5655971""], ""bases"":...",1,PRJNA774626,plasma,male,Colon Cancer


In [8]:
# --- Step 8: Alpha Diversity Analysis (wide-native manager input) ---
#
# Uses analysis_wide_df directly. No long-format reconstruction is performed.
from managers.AlphaDiversityManager.alpha_diversity_manager import AlphaDiversityManager

if 'analysis_wide_df' not in locals() or 'taxon_lookup' not in locals() or 'metadata_cols' not in locals():
    raise RuntimeError("Run Step 7 first to create analysis_wide_df")

alpha_manager = AlphaDiversityManager(
    analysis_wide_df=analysis_wide_df,
    taxon_lookup_df=taxon_lookup,
    metadata_cols=metadata_cols
)
alpha_manager.run()

📊 Building taxonomy lineage database...
Building lineage database from 4233 rows...
✅ Built lineage database with 4233 unique taxa
📋 Standard ranks available: ['superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
   (+ 25 additional ranks available)


In [9]:
# --- Step 8: Taxonomic Abundance Analysis (wide-native manager input) ---
#
# Uses analysis_wide_df directly. No long-format reconstruction is performed.
from managers.TaxonomicAbundanceManager import TaxonomicAbundanceManager

if 'analysis_wide_df' not in locals() or 'taxon_lookup' not in locals() or 'metadata_cols' not in locals():
    raise RuntimeError("Run Step 7 first to create analysis_wide_df")

abundance_manager = TaxonomicAbundanceManager(
    analysis_wide_df=analysis_wide_df,
    taxon_lookup_df=taxon_lookup,
    metadata_cols=metadata_cols
)
abundance_manager.run()

In [10]:
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

if 'analysis_wide_df' in locals() and 'metadata_cols' in locals() and 'taxon_lookup' in locals():

    updated_meta_df = analysis_wide_df[metadata_cols].copy()
    taxon_cols = [c for c in analysis_wide_df.columns if c not in metadata_cols]

    # Build a sample x taxon count matrix from the canonical wide dataset.
    sample_taxa_counts = analysis_wide_df[['acc'] + taxon_cols].set_index('acc').copy()
    sample_totals = sample_taxa_counts.sum(axis=1)
    safe_totals = sample_totals.replace(0, 1)

    # Convert to CPM using total counts per sample.
    cpm_matrix = sample_taxa_counts.div(safe_totals, axis=0) * 1_000_000

    # Human-readable labels for taxa shown in the selector.
    lookup = taxon_lookup.copy()
    lookup = lookup.dropna(subset=['taxon_key'])
    lookup['taxon_key'] = lookup['taxon_key'].astype(str)
    lookup['rank'] = lookup['rank'].astype(str)
    lookup['name'] = lookup['name'].astype(str)
    lookup['taxon_label'] = lookup['rank'] + ': ' + lookup['name']
    lookup = lookup[lookup['taxon_key'].isin(cpm_matrix.columns)]

    # Ensure deterministic string-only selector options to avoid mixed-type sort errors.
    label_to_key = {str(label): str(key) for label, key in zip(lookup['taxon_label'], lookup['taxon_key'])}
    taxa_options = sorted(label_to_key.keys(), key=lambda x: str(x))

    exclude_cols = ['acc']
    meta_cols = [
        col for col in updated_meta_df.columns
        if col not in exclude_cols and 1 < updated_meta_df[col].nunique(dropna=True) < 50
    ]

    taxon_selector = widgets.Combobox(
        placeholder='Start typing taxon (rank: name)...',
        options=taxa_options,
        description='Taxon:',
        ensure_option=True
    )
    condition_dropdown = widgets.Dropdown(options=sorted(meta_cols), description='Condition:')
    group1_dropdown = widgets.Dropdown(description='Group A:')
    group2_dropdown = widgets.Dropdown(description='Group B:')
    run_button = widgets.Button(description='Run Mann-Whitney', button_style='info')
    output_area = widgets.Output()

    def update_groups(*args):
        selected_col = condition_dropdown.value
        if selected_col and selected_col in updated_meta_df.columns:
            unique_vals = sorted(updated_meta_df[selected_col].dropna().astype(str).unique())
            group1_dropdown.options = unique_vals
            group2_dropdown.options = unique_vals
            if len(unique_vals) > 1:
                group1_dropdown.value = unique_vals[0]
                group2_dropdown.value = unique_vals[1]

    condition_dropdown.observe(update_groups, 'value')
    update_groups()

    def analyze_taxon(b):
        with output_area:
            clear_output(wait=True)
            selected_label = taxon_selector.value
            condition = condition_dropdown.value
            g1 = group1_dropdown.value
            g2 = group2_dropdown.value

            if not selected_label or selected_label not in label_to_key:
                print('Please select a valid taxon label from the list.')
                return

            taxon_key = label_to_key[selected_label]
            taxon_name = selected_label

            taxon_cpm = cpm_matrix[[taxon_key]].reset_index()
            taxon_cpm.columns = ['acc', 'cpm_value']

            merged_data = pd.merge(
                taxon_cpm,
                updated_meta_df[['acc', condition]],
                on='acc',
                how='inner'
            )
            merged_data[condition] = merged_data[condition].astype(str)

            data_g1 = merged_data[merged_data[condition] == g1]['cpm_value']
            data_g2 = merged_data[merged_data[condition] == g2]['cpm_value']

            if len(data_g1) == 0 or len(data_g2) == 0:
                print('One of the groups has no samples after filtering. Choose another grouping.')
                return

            _, p_value = stats.mannwhitneyu(data_g1, data_g2, alternative='two-sided')

            p_text = 'p < 0.001' if p_value < 0.001 else f"p = {p_value:.3f}"

            plt.figure(figsize=(8, 5))
            plot_data = merged_data[merged_data[condition].isin([g1, g2])]
            sns.boxplot(
                data=plot_data,
                x=condition,
                y='cpm_value',
                hue=condition,
                palette='Set2',
                showfliers=False,
                legend=False
            )
            sns.stripplot(
                data=plot_data,
                x=condition,
                y='cpm_value',
                color='.25',
                alpha=0.6,
                jitter=True
            )

            plt.title(f'{taxon_name} Abundance\nMann-Whitney U: {p_text}')
            plt.ylabel('Abundance (Counts Per Million - CPM)')
            plt.xlabel(condition)
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()

            print('-' * 40)
            print(f'Taxon: {taxon_name}')
            print(f'Comparison: {g1} (n={len(data_g1)}) vs {g2} (n={len(data_g2)})')
            print(f'P-value: {p_text} (Raw: {p_value:.6f})')

    run_button.on_click(analyze_taxon)
    ui_controls = widgets.VBox([
        taxon_selector,
        widgets.HBox([condition_dropdown, group1_dropdown, group2_dropdown]),
        run_button
    ])
    display(ui_controls, output_area)
else:
    print('Run Step 7 first to create analysis_wide_df, metadata_cols, and taxon_lookup.')

Output()

In [11]:
# --- Step 9: Longitudinal Analysis ---
#
# NOTE: LongitudinalAnalysisManager needs to be refactored to accept
# (taxonomy_df, metadata_df) separately, following the same pattern as
# AlphaDiversityManager and TaxonomicAbundanceManager.
# This is tracked as a follow-up task.
#
# from managers.LongitudinalAnalysisManager import LongitudinalAnalysisManager
# longitudinal_manager = LongitudinalAnalysisManager(taxonomy_df=taxonomy_df, metadata_df=enriched_metadata_df)
# longitudinal_manager.run()
